In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_data = pd.read_csv('/content/train.csv')
train_data.drop(columns='Location', inplace=True)
train_data.head()

In [ ]:
invalid_values = ['#', '*', 'x', 'A']
time_col = train_data.columns[2:]

for col in time_col:
  mask = train_data[col].isin(invalid_values)
  if mask.all():
        train_data[col] = train_data[col].bfill()
  else:
      train_data[col].mask(mask, inplace=True)
      train_data[col] = train_data[col].bfill().ffill()

train_data[time_col] = train_data[time_col].apply(pd.to_numeric, errors='coerce')

In [ ]:
# Check if there are any NaN values in the DataFrame
nan_exists = train_data.isna().any().any()

# Display the rows with NaN values if any exist
nan_rows = None
if nan_exists:
    nan_rows = train_data[train_data.isna().any(axis=1)]

nan_exists, nan_rows

In [ ]:
month_data_pm25 = {}
PM25_index_step = 18
PM25_row_index = 9

for month_index in range(12):
    # 存該月的 PM2.5 數據
    data_matrix_pm25 = np.zeros((1, 20 * 24), dtype=np.float32)  # 只有一行，只關心 PM2.5

    for day in range(20):
        for hour in range(24):
            row_index = PM25_row_index + PM25_index_step * (month_index * 20 + day)

            # 抽取相應的 PM2.5 數據並賦值到數據矩陣中
            data_matrix_pm25[0, day * 24 + hour] = train_data.iloc[row_index, 2 + hour]

    # 將該月的 PM2.5 數據矩陣儲存在字典中
    month_data_pm25[month_index] = data_matrix_pm25

In [ ]:
# 連接所有月份的 PM2.5 數據
all_pm25_data = np.concatenate([month_data_pm25[month][0, :] for month in range(12)])

time_window = 9
X_train = []
y_train = []

for i in range(len(all_pm25_data) - time_window):
    X_train.append(all_pm25_data[i:i+time_window])
    y_train.append(all_pm25_data[i+time_window])

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32).reshape(-1, 1)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('First 10 rows of X_train:')
print(X_train[:10, :])
print('First 10 rows of y_train:')
print(y_train[:10, :])

In [ ]:
# normalization
X_mean = np.mean(X_train, axis=0)
X_std = np.std(X_train, axis=0)

X_train_normalized = (X_train - X_mean) / X_std

# 確保標準差不為零以避免除以零的錯誤
X_std[X_std == 0] = 1
X_train_normalized = (X_train - X_mean) / X_std

In [ ]:
total_samples = X_train.shape[0]
train_size = int(total_samples * 0.8)
val_size = total_samples - train_size

X_train_split = X_train[:train_size]
y_train_split = y_train[:train_size]

X_val_split = X_train[train_size:]
y_val_split = y_train[train_size:]

print('Training set:', X_train_split.shape, y_train_split.shape)
print('Validation set:', X_val_split.shape, y_val_split.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LinearRegression:
    def __init__(self, learning_rate=0.01, epochs=1000, reg_lambda=0.01, lr_decay=0.999):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.reg_lambda = reg_lambda
        self.lr_decay = lr_decay
        self.weights = None
        self.train_loss_history = []
        self.val_loss_history = []

    def fit(self, X_train, y_train, X_val=None, y_val=None, method='gradient_descent'):
        # 確保 y_train 和 y_val 是列向量
        y_train = y_train.reshape(-1, 1)
        if y_val is not None:
            y_val = y_val.reshape(-1, 1)

        if method == 'gradient_descent':
            self._gradient_descent(X_train, y_train, X_val, y_val)
        elif method == 'closed_form':
            self._closed_form(X_train, y_train)

    def _gradient_descent(self, X_train, y_train, X_val=None, y_val=None):
        # 初始化權重(包括 bias)
        self.weights = np.zeros((X_train.shape[1] + 1, 1))

        # 添加 bias
        X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
        X_val_b = np.c_[np.ones((X_val.shape[0], 1)), X_val] if X_val is not None else None

        for i in range(self.epochs):
            y_pred = X_train_b.dot(self.weights)
            errors = y_train - y_pred

            gradients = -2 * X_train_b.T.dot(errors) / X_train.shape[0] + 2 * self.reg_lambda * self.weights
            self.weights -= self.learning_rate * gradients

            # Training loss
            train_loss = np.sqrt(np.mean(errors ** 2))
            self.train_loss_history.append(train_loss)

            # Validation loss
            if X_val_b is not None:
                val_errors = y_val - X_val_b.dot(self.weights)
                val_loss = np.sqrt(np.mean(val_errors ** 2))
                self.val_loss_history.append(val_loss)

            # 每5000个epoch印一次loss
            if i % 5000 == 0:
                print(f"Epoch {i}: Training RMSE = {train_loss}, Validation RMSE = {val_loss if X_val is not None else 'N/A'}")

            self.learning_rate *= self.lr_decay

    def _closed_form(self, X_train, y_train):
        X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
        A = X_train_b.T.dot(X_train_b) + self.reg_lambda * np.eye(X_train_b.shape[1])
        b = X_train_b.T.dot(y_train)
        self.weights = np.linalg.solve(A, b)

        y_pred = X_train_b.dot(self.weights)
        train_loss = np.sqrt(np.mean((y_train - y_pred) ** 2))
        print(f"Closed-form Training RMSE: {train_loss}")

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        # return X_b.dot(self.weights)
        predictions = X_b.dot(self.weights)
        return predictions.flatten()

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        return np.sqrt(np.mean((y_pred - y) ** 2))

    def plot_losses(self):
        plt.plot(self.train_loss_history, label='Training RMSE')
        if self.val_loss_history:
            plt.plot(self.val_loss_history, label='Validation RMSE')
        plt.title('RMSE Loss Over Epochs')
        plt.xlabel('Epochs')
        plt.ylabel('RMSE')
        plt.legend()
        plt.show()

In [ ]:
model = LinearRegression(learning_rate=0.0001, epochs=30000, reg_lambda=0.01, lr_decay=1)
model.fit(X_train_split, y_train_split, X_val_split, y_val_split, method='gradient_descent')
model.plot_losses()

In [ ]:
model = LinearRegression(learning_rate=0.01, epochs=30000, reg_lambda=0.1, lr_decay=0.99)
model.fit(X_train_normalized, y_train, method='closed_form')

In [ ]:
test_data = pd.read_csv('/content/test.csv', header = None)
test_data.head()

In [ ]:
invalid_values = ['#', '*', 'x', 'A']
test_time_col = test_data.columns[2:]

for col in test_time_col:
  mask = test_data[col].isin(invalid_values)
  if mask.all():
        test_data[col] = test_data[col].bfill()
  else:
      test_data[col].mask(mask, inplace=True)
      test_data[col] = test_data[col].bfill().ffill()

test_data[test_time_col] = test_data[test_time_col].apply(pd.to_numeric, errors='coerce')

In [ ]:
# Check if there are any NaN values in the DataFrame
nan_exists = test_data.isna().any().any()

# Display the rows with NaN values if any exist
nan_rows = None
if nan_exists:
    nan_rows = test_data[test_data.isna().any(axis=1)]

nan_exists, nan_rows

In [ ]:
row, col = test_data.shape
num_samples = int(row / 18)
X_test = np.zeros((num_samples, col-2), dtype=np.float32)  # 減去前兩列

# 提取每18行的第10行（index為9）
for i in range(num_samples):
    X_test[i] = test_data.iloc[i*18 + 9, 2:].values

print(X_test.shape)

In [ ]:
X_test_normalized = (X_test - X_mean) / X_std

# 確保標準差不為零以避免除以零的錯誤
X_std[X_std == 0] = 1
X_test_normalized = (X_test - X_mean) / X_std

In [ ]:
y_test_pred = model.predict(X_test_normalized)
prediction = [[f'index_{i}', y_test_pred[i]] for i in range(len(y_test_pred))]
prediction = pd.DataFrame(prediction, columns=['index', 'answer'])
prediction.to_csv('predict.csv', index=False)